In [21]:
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from peft import LoraConfig, get_peft_model
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

import pandas as pd
from datasets import Dataset

In [25]:
dataset = Dataset.from_csv("dataset_final.csv")

In [26]:
# Cek struktur dataset Anda
print(f"Jumlah samples: {len(dataset)}")
print(f"Sample pertama:\n{dataset[0]['text']}")

# Reset indeks dataset
dataset = dataset.shuffle(seed=42)
dataset = dataset.select(range(len(dataset)))  

Jumlah samples: 1347
Sample pertama:
### Jenis Sel: basophil ### Penjelasan: Sel basofil segmen menunjukkan granula sitoplasma yang sangat padat dan berwarna gelap yang menutupi sebagian besar inti sel dengan bentuk lobus tidak beraturan. Kondisi morfologi ini dapat mengindikasikan aktivasi abnormal lini granulosit yang sering ditemukan pada leukemia mieloid kronis maupun akut. Pemeriksaan darah tepi dan analisis sumsum tulang sangat dianjurkan untuk evaluasi lebih lanjut. <|endoftext|>


In [27]:
model = AutoModelForCausalLM.from_pretrained(
    "flax-community/gpt2-small-indonesian",
    use_safetensors=True, 
    ignore_mismatched_sizes=True
)
tokenizer = AutoTokenizer.from_pretrained("flax-community/gpt2-small-indonesian")
print(model)

'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 885b8ebd-2015-46a1-86d9-5c83a3d89524)')' thrown while requesting HEAD https://huggingface.co/flax-community/gpt2-small-indonesian/resolve/main/config.json
Retrying in 1s [Retry 1/5].


GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.0, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.0, inplace=False)
          (resid_dropout): Dropout(p=0.0, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.0, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)


In [28]:
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [29]:
lora_config = LoraConfig(
    r=8,  # rank
    lora_alpha=32,  
    target_modules=["c_attn", "c_proj", "c_fc"],  
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM"
)

In [30]:
model = get_peft_model(model, lora_config)

d:\MiniConda\envs\torch_gpu\Lib\site-packages\peft\tuners\lora\layer.py:2504: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


In [31]:
def tokenize_function(examples):
    # Ini yang mengubah teks jadi input_ids dan attention_mask
    return tokenizer(
        examples["text"], 
        truncation=True, 
        padding="max_length", 
        max_length=300
    )

In [32]:
tokenized_dataset = dataset.map(tokenize_function, batched=True, remove_columns=["text"])

# 3. Cek struktur (opsional untuk debugging)
print(f"Jumlah samples: {len(tokenized_dataset)}")
print(f"Fitur dataset: {tokenized_dataset.column_names}")

Map:   0%|          | 0/1347 [00:00<?, ? examples/s]

Jumlah samples: 1347
Fitur dataset: ['input_ids', 'attention_mask']


In [33]:
training_args = TrainingArguments(
    output_dir="./gpt2-indonesian-lora",
    num_train_epochs=10,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=10,
    save_strategy="epoch",
    learning_rate=2e-4,
    fp16=True,  
)


In [34]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset, # Pakai yang sudah di-tokenize!
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
)

In [35]:
trainer.train()

Step,Training Loss
10,4.323300
20,4.398300
30,4.458700
40,4.349500
50,4.409800
60,4.341600
70,4.426000
80,4.328000
90,4.404600
100,4.268000


'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 21090678-b564-4501-9031-c52575e82f1d)')' thrown while requesting HEAD https://huggingface.co/flax-community/gpt2-small-indonesian/resolve/main/config.json
Retrying in 1s [Retry 1/5].
'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: ecc59ac9-ba20-4de7-8cb0-1c6c33a50145)')' thrown while requesting HEAD https://huggingface.co/flax-community/gpt2-small-indonesian/resolve/main/config.json
Retrying in 1s [Retry 1/5].
'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 5b9bde05-e2bf-4717-b6ee-f215e941d6ff)')' thrown while requesting HEAD https://huggingface.co/flax-community/gpt2-small-indonesian/resolve/main/config.json
Retrying in 1s [Retry 1/5].
'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Re

TrainOutput(global_step=3370, training_loss=1.8600041403614802, metrics={'train_runtime': 1110.6099, 'train_samples_per_second': 12.128, 'train_steps_per_second': 3.034, 'total_flos': 2090869521408000.0, 'train_loss': 1.8600041403614802, 'epoch': 10.0})

In [36]:
trainer.save_model("./models")

# 2. Simpan Tokenizer (Kamusnya)
tokenizer.save_pretrained("./models")

print("--- BERHASIL! Model sudah aman di folder 'model_darah_zahran_final' ---")

--- BERHASIL! Model sudah aman di folder 'model_darah_zahran_final' ---
